# FRAG-SED Benchmark: Colab GPU Batch Inference (`Qwen/Qwen3-8B`)

This notebook runs the large-scale unbounded **LSED Official (ACL'25)** LLM tweet-rewriting baseline on `Event2012 (M_00..M_05)`:
1. **LSED Official (`M_01..M_05`, `N` = 15,892 tweet rewrites, `max_tokens=256`)**
2. **LSED M_00 Training Summaries (`M_00`, `N` = 20,254 tweet rewrites, `max_tokens=256`, REQUIRED for aligned training)**

Note: `Cequel` and `COBRAS` run under the **20k tokens/block** budget and no longer require unbounded Colab batch inference.

Settings match our local benchmark exactly: `Qwen/Qwen3-8B`, `temperature=0.0`, `enable_thinking=False`.
Select **Runtime -> Change runtime type -> GPU** (L4/A100 recommended; T4 15GB also supported via 4-bit quantization), then **Run all**.

In [ ]:
from google.colab import drive
import os, torch

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/FRAG_SED_Experiments'
IN_DIR = os.path.join(BASE_DIR, 'colab_inputs')
OUT_DIR = os.path.join(BASE_DIR, 'colab_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU detected! Please switch Colab Runtime to GPU (T4/L4/A100).'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}, VRAM: {vram_gb:.1f} GB')
print('Input files:', os.listdir(IN_DIR))

In [ ]:
# Cell 2: Install & load Qwen/Qwen3-8B (vLLM on >=22GB L4/A100; 4-bit Transformers on T4)
MODEL_ID = 'Qwen/Qwen3-8B'
USE_VLLM = vram_gb >= 22.0

if USE_VLLM:
    !pip install -q vllm>=0.8.5
    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    llm = LLM(model=MODEL_ID, dtype='bfloat16', max_model_len=1024, gpu_memory_utilization=0.90)
    print('Loaded vLLM bf16 backend.')
else:
    !pip install -q transformers>=4.51.0 accelerate bitsandbytes
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map='auto')
    model.eval()
    print('Loaded 4-bit Transformers backend for T4.')

In [ ]:
# Cell 3: Helper to run JSONL prompts with resume + exact token counts
import json, time
from tqdm.auto import tqdm

def format_prompt(rec):
    msgs = []
    if rec.get('sys_msg'):
        msgs.append({'role': 'system', 'content': rec['sys_msg']})
    msgs.append({'role': 'user', 'content': rec['user_msg']})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def run_jsonl_task(in_filename, out_filename, max_new_tokens, batch_size=24):
    in_path = os.path.join(IN_DIR, in_filename)
    out_path = os.path.join(OUT_DIR, out_filename)
    with open(in_path, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f if line.strip()]

    done_ids = set()
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        done_ids.add(json.loads(line)['request_id'])
                    except Exception:
                        pass
    todo = [r for r in items if r['request_id'] not in done_ids]
    print(f'{in_filename}: total={len(items)}, already_done={len(done_ids)}, remaining={len(todo)}')
    if not todo:
        return

    t0 = time.time()
    with open(out_path, 'a', encoding='utf-8') as fout:
        if USE_VLLM:
            chunk_sz = 512
            sp = SamplingParams(temperature=0.0, max_tokens=max_new_tokens)
            for s in tqdm(range(0, len(todo), chunk_sz), desc=in_filename):
                chunk = todo[s : s + chunk_sz]
                prompts = [format_prompt(r) for r in chunk]
                outs = llm.generate(prompts, sp, use_tqdm=False)
                for r, out in zip(chunk, outs):
                    txt = out.outputs[0].text.strip()
                    pt = len(out.prompt_token_ids)
                    ct = len(out.outputs[0].token_ids)
                    fout.write(json.dumps({
                        'request_id': r['request_id'],
                        'block_id': r['block_id'],
                        'text': txt,
                        'prompt_tokens': pt,
                        'completion_tokens': ct,
                    }, ensure_ascii=False) + '\n')
                fout.flush()
        else:
            for s in tqdm(range(0, len(todo), batch_size), desc=in_filename):
                chunk = todo[s : s + batch_size]
                prompts = [format_prompt(r) for r in chunk]
                enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=896).to(model.device)
                in_len = enc['input_ids'].shape[1]
                with torch.no_grad():
                    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
                for bi, r in enumerate(chunk):
                    pt = int(enc['attention_mask'][bi].sum().item())
                    out_ids = gen[bi, in_len:]
                    # Count up to first eos if present
                    eos_pos = (out_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    if len(eos_pos) > 0:
                        out_ids = out_ids[: int(eos_pos[0].item()) + 1]
                    ct = int(len(out_ids))
                    txt = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
                    fout.write(json.dumps({
                        'request_id': r['request_id'],
                        'block_id': r['block_id'],
                        'text': txt,
                        'prompt_tokens': pt,
                        'completion_tokens': ct,
                    }, ensure_ascii=False) + '\n')
                if (s // batch_size) % 10 == 0:
                    fout.flush()
            fout.flush()
    print(f'Finished {in_filename} in {time.time() - t0:.1f}s -> {out_path}')

In [ ]:
# Cell 4: Auto-detect LSED validation/dev prompts (M_02..M_05 or M_01..M_05) and REQUIRED LSED M_00 (20,254 tweets)
for p_name in ['lsed_M02_M05_prompts.jsonl', 'lsed_M01_M05_prompts.jsonl']:
    if os.path.exists(os.path.join(IN_DIR, p_name)):
        run_jsonl_task(p_name, p_name.replace('_prompts.jsonl', '_responses.jsonl'), max_new_tokens=256, batch_size=24)
run_jsonl_task('lsed_M00_prompts.jsonl', 'lsed_M00_responses.jsonl', max_new_tokens=256, batch_size=24)

In [ ]:
# Cell 5: Verify 100% completion
for name in ['lsed_M02_M05_responses.jsonl', 'lsed_M01_M05_responses.jsonl', 'lsed_M00_responses.jsonl']:
    p = os.path.join(OUT_DIR, name)
    cnt = sum(1 for _ in open(p, 'r', encoding='utf-8')) if os.path.exists(p) else 0
    print(name, 'rows =', cnt)